### Постановка задачи

Решаем уравнение Лапласа для потенциала
$$\nabla^2 \varphi(x,y) = 0,$$
в прямоугольной области
$$0 \le x \le L_x,\quad 0 \le y \le L_y.$$

Граничные условия (параллельные пластины):
- нижняя пластина:  $\varphi(x,0) = -V$;
- верхняя пластина: $\varphi(x,L_y) = +V$;
- боковые стенки:   $\varphi(0,y) = \varphi(L_x,y) = 0.$

Используем равномерную сетку



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

Lx, Ly = 1.0, 1.0
Nx, Ny = 81, 81
V = 1.0  # модуль потенциала пластин

dx = Lx / (Nx - 1)
dy = Ly / (Ny - 1)

x = np.linspace(0.0, Lx, Nx)
y = np.linspace(0.0, Ly, Ny)

X, Y = np.meshgrid(x, y, indexing='xy')

inv_dxsq = 1.0 / (dx * dx)
inv_dysq = 1.0 / (dy * dy)
denom = 2.0 * (inv_dxsq + inv_dysq)

tol = 1e-6
max_iters = 20000


### Геометрия и численные методы

Внутрь конденсатора можно поместить проводящий квадрат, для которого
потенциал постоянен:
$$\varphi(x,y) = \varphi_\text{cond} = \text{const}.$$

Для численного решения используем конечно-разностную аппроксимацию:
$$
\varphi_{i,j}^{(n+1)} =
\frac{
(\varphi_{i+1,j}^{(*)} + \varphi_{i-1,j}^{(*)})/dx^2 +
(\varphi_{i,j+1}^{(*)} + \varphi_{i,j-1}^{(*)})/dy^2
}{
2\,(1/dx^2 + 1/dy^2)
},
$$

где:
- в методе **Якоби** везде используется старый слой $\varphi^{(n)}$;
- в методе **Гаусса–Зейделя** справа берутся уже обновлённые значения.

Поле напряжённости:
$$ \mathbf{E} = -\nabla \varphi, \quad
E_x = -\frac{\partial \varphi}{\partial x},\;
E_y = -\frac{\partial \varphi}{\partial y}.
$$


In [ ]:
def make_geometry(with_square=False):
    """
    Создаём начальное поле потенциала и маску фиксированных узлов.
    """
    phi = np.zeros((Ny, Nx), dtype=float)
    fixed = np.zeros_like(phi, dtype=bool)

    # --- граничные условия: параллельные пластины ---
    phi[0, :]  = -V   # нижняя пластина
    phi[-1, :] = +V   # верхняя пластина
    phi[:, 0]  = 0.0  # левая боковая стенка
    phi[:, -1] = 0.0  # правая боковая стенка

    fixed[0, :]  = True
    fixed[-1, :] = True
    fixed[:, 0]  = True
    fixed[:, -1] = True

    square = None
    if with_square:
        i1 = Nx // 3
        i2 = 2 * Nx // 3
        j1 = Ny // 3
        j2 = 2 * Ny // 3

        phi[j1:j2+1, i1:i2+1] = 0.0
        fixed[j1:j2+1, i1:i2+1] = True

        square = (i1, i2, j1, j2)

    return phi, fixed, square


def solve_laplace_jacobi(phi_init, fixed, max_iters=max_iters, tol=tol):
    """
    Решение уравнения Лапласа методом Якоби.
    """
    phi = phi_init.copy()
    Ny_loc, Nx_loc = phi.shape
    changes = []

    for it in range(1, max_iters + 1):
        phi_old = phi.copy()
        max_change = 0.0

        for j in range(1, Ny_loc - 1):
            for i in range(1, Nx_loc - 1):
                # пропускаем узлы с фиксированным потенциалом
                if fixed[j, i]:
                    continue

                new_val = ((phi_old[j, i + 1] + phi_old[j, i - 1]) * inv_dxsq +
                           (phi_old[j + 1, i] + phi_old[j - 1, i]) * inv_dysq) / denom

                diff = abs(new_val - phi_old[j, i])
                if diff > max_change:
                    max_change = diff

                phi[j, i] = new_val

        changes.append(max_change)
        if max_change < tol:
            break

    print(f'Якоби: итераций = {it}, финальное max|Δφ| = {changes[-1]:.2e}')
    return phi, np.array(changes)


def solve_laplace_gs(phi_init, fixed, max_iters=max_iters, tol=tol):
    """
    Решение уравнения Лапласа методом Гаусса–Зейделя.
    """
    phi = phi_init.copy()
    Ny_loc, Nx_loc = phi.shape
    changes = []

    for it in range(1, max_iters + 1):
        max_change = 0.0

        for j in range(1, Ny_loc - 1):
            for i in range(1, Nx_loc - 1):
                if fixed[j, i]:
                    continue

                old_val = phi[j, i]
                new_val = ((phi[j, i + 1] + phi[j, i - 1]) * inv_dxsq +
                           (phi[j + 1, i] + phi[j - 1, i]) * inv_dysq) / denom

                diff = abs(new_val - old_val)
                if diff > max_change:
                    max_change = diff

                phi[j, i] = new_val

        changes.append(max_change)
        if max_change < tol:
            break

    print(f'Гаусс–Зейдель: итераций = {it}, финальное max|Δφ| = {changes[-1]:.2e}')
    return phi, np.array(changes)


def compute_E(phi):
    """
    Вычисляем поле E = -grad φ с помощью конечных разностей.
    """
    Ex = np.zeros_like(phi)
    Ey = np.zeros_like(phi)

    # внутренние точки (центральные разности)
    Ex[:, 1:-1] = -(phi[:, 2:] - phi[:, :-2]) / (2 * dx)
    Ey[1:-1, :] = -(phi[2:, :] - phi[:-2, :]) / (2 * dy)

    # края (односторонние разности)
    Ex[:, 0]  = -(phi[:, 1] - phi[:, 0]) / dx
    Ex[:, -1] = -(phi[:, -1] - phi[:, -2]) / dx
    Ey[0, :]  = -(phi[1, :] - phi[0, :]) / dy
    Ey[-1, :] = -(phi[-1, :] - phi[-2, :]) / dy

    return Ex, Ey


### Конденсатор без внутреннего проводника

Решаем уравнение Лапласа с указанными граничными условиями только для
пластин, без препятствий.

Вычисляем:
- распределение потенциала $\varphi(x,y)$;
- поле $\mathbf{E} = -\nabla \varphi$;
- график сходимости $\max|\Delta \varphi^{(n)}|$ для метода Гаусса–Зейделя.


In [ ]:
phi0_plain, fixed_plain, _ = make_geometry(with_square=False)

phi_plain_gs, changes_plain_gs = solve_laplace_gs(phi0_plain, fixed_plain)

Ex_plain, Ey_plain = compute_E(phi_plain_gs)

# Потенциал
plt.figure(figsize=(6, 5))
im = plt.imshow(phi_plain_gs, extent=[0, Lx, 0, Ly],
                origin='lower', aspect='equal')
plt.colorbar(im, label='φ')
plt.title('Потенциал между пластинами (без квадрата)')
plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()

# Векторное поле E
plt.figure(figsize=(6, 5))
skip = max(1, Nx // 32)
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip],
           Ex_plain[::skip, ::skip], Ey_plain[::skip, ::skip],
           scale=40)
plt.title('Поле E между пластинами (без квадрата)')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.tight_layout()

# Сходимость метода Гаусса–Зейделя
plt.figure(figsize=(6, 4))
plt.semilogy(np.arange(1, len(changes_plain_gs) + 1), changes_plain_gs)
plt.xlabel('Итерация')
plt.ylabel('max |Δφ|')
plt.title('Сходимость метода Гаусса–Зейделя (без квадрата)')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.tight_layout()

plt.show()

###  Конденсатор с проводящим квадратом

Теперь внутри области задаём область проводника с постоянным
потенциалом:
$$\varphi(x,y) = \varphi_\text{cond} = 0.$$

Внутри проводника поле должно быть близко к нулю,
а силовые линии $\mathbf{E}$ огибают квадрат.  
Решение опять строим методом Гаусса–Зейделя.


In [ ]:
phi0_square, fixed_square, square = make_geometry(with_square=True)

phi_square_gs, changes_square_gs = solve_laplace_gs(phi0_square, fixed_square)

Ex_square, Ey_square = compute_E(phi_square_gs)

plt.figure(figsize=(6, 5))
im = plt.imshow(phi_square_gs, extent=[0, Lx, 0, Ly],
                origin='lower', aspect='equal')
plt.colorbar(im, label='φ')
plt.title('Потенциал между пластинами с проводящим квадратом')
plt.xlabel('x')
plt.ylabel('y')

if square is not None:
    i1, i2, j1, j2 = square
    x1, x2 = x[i1], x[i2]
    y1, y2 = y[j1], y[j2]
    plt.plot([x1, x2, x2, x1, x1],
             [y1, y1, y2, y2, y1],
             linewidth=2, color='k')

plt.tight_layout()

plt.figure(figsize=(6, 5))
skip = max(1, Nx // 32)
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip],
           Ex_square[::skip, ::skip], Ey_square[::skip, ::skip],
           scale=40)
plt.title('Поле E с проводящим квадратом')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')

if square is not None:
    i1, i2, j1, j2 = square
    x1, x2 = x[i1], x[i2]
    y1, y2 = y[j1], y[j2]
    plt.plot([x1, x2, x2, x1, x1],
             [y1, y1, y2, y2, y1],
             linewidth=2)

plt.tight_layout()

plt.show()


### Сравнение сходимости Якоби и Гаусса–Зейделя

Сравниваем скорость сходимости двух схем для случая с проводящим квадратом.

График показывает зависимость
$$ \max_{i,j} |\Delta \varphi_{i,j}^{(n)}| $$
от номера итерации $n$.



In [ ]:
phi_square_jacobi, changes_square_jacobi = solve_laplace_jacobi(phi0_square, fixed_square)
phi_square_gs_cmp, changes_square_gs_cmp = solve_laplace_gs(phi0_square, fixed_square)

plt.figure(figsize=(7, 5))
plt.semilogy(np.arange(1, len(changes_square_jacobi) + 1),
             changes_square_jacobi, label='Якоби')
plt.semilogy(np.arange(1, len(changes_square_gs_cmp) + 1),
             changes_square_gs_cmp, label='Гаусс–Зейдель')

plt.xlabel('Итерация')
plt.ylabel('max |Δφ|')
plt.title('Сравнение сходимости Якоби и Гаусса–Зейделя (с проводящим квадратом)')
plt.grid(True, which='both', linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()

plt.show()
